# 03 · RAG over the CRR

**Session 5 — Agent Architecture and Ecosystem Tools · MBRF Data Analytics**

A model does not know your documents, and even when it knows a regulation it cannot show you **where** an answer comes from. *Retrieval-augmented generation* (RAG) fixes both: before answering, retrieve the relevant passages and give them to the model as context.

1. The corpus: the Capital Requirements Regulation (CRR), consolidated text.
2. **Chunking**: cut documents into pieces that can be retrieved.
3. **Embeddings**: turn text into vectors that capture meaning.
4. **Search**: semantic, keyword, and hybrid — and how to tell which works better.
5. **Generation** with citations.
6. RAG as a **tool** of the agent from notebook 02.

## 0. Setup

In [ ]:
%pip install -q git+https://github.com/albertotb/curso-agentes-ia
# Without git: %pip install -q https://github.com/albertotb/curso-agentes-ia/archive/refs/heads/main.zip

In [ ]:
import re
import time

import numpy as np
import pandas as pd

from cai import utils

PROVIDER = "openai"  # "openai" | "gemini"
MODEL = utils.DEFAULT_MODELS[PROVIDER]
EMBEDDING_MODEL = utils.EMBEDDING_MODELS[PROVIDER]
client = utils.get_client(PROVIDER)
print("Using", PROVIDER, MODEL, EMBEDDING_MODEL)

## 1. Why retrieval?

Ask the model directly. It may well give the right figure — the CRR is public and was in its training data — but check what you **cannot** do with the answer.

In [ ]:
question = "Under the CRR, which percentage applies to the output floor once fully phased in, and in which article is it set?"
answer = client.chat.completions.create(model=MODEL, messages=[{"role": "user", "content": question}])
print(answer.choices[0].message.content)

- Is that the **current** consolidated text, or the version the model saw during training?
- Is the article number right, or just plausible? You cannot click on it.
- And for your bank's **internal** policies, procedures or inspection reports, the model has nothing at all.

RAG: *retrieve* the passages, *augment* the prompt with them, *generate* an answer that cites them.

## 2. The corpus

The CRR (Regulation (EU) No 575/2013), consolidated text of 26 June 2026, one record per article.

In [ ]:
articles = utils.load_crr_articles()
df = pd.DataFrame(articles)
df["chars"] = df["text"].str.len()
print(len(df), "articles,", df["chars"].sum(), "characters (~", df["chars"].sum() // 4, "tokens)")
df.groupby("part")["chars"].agg(["count", "sum", "median", "max"])

In [ ]:
print(df.set_index("article").loc["92", "text"][:1200])

## 3. Chunking

We retrieve **pieces** of documents, not whole documents: a 60,000-character article (Article 4, the definitions) is useless as context, and too much text dilutes the answer and costs money.

The naive way is to cut every N characters:

In [ ]:
text_92 = df.set_index("article").loc["92", "text"]
naive_chunks = [text_92[i : i + 500] for i in range(0, len(text_92), 500)]
for c in naive_chunks[:3]:
    print(repr(c[-120:]), "\n")

Sentences, lists and paragraphs are cut in half, and a piece like `"…a total capital ratio of 8 %;"` no longer says which regulation or article it belongs to.

Legal text has a structure we can use: **article → numbered paragraphs**. We keep each article whole when it is short, split long ones at paragraph boundaries, and prepend a **header** to every chunk so it still says where it comes from.

In [ ]:
MAX_CHARS = 3000


def split_article(text: str, max_chars: int = MAX_CHARS) -> list[str]:
    """Group consecutive numbered paragraphs ("1.", "2a.") into pieces of at most max_chars."""
    pieces, current = [], ""
    for paragraph in re.split(r"\n(?=\d+[a-z]*\. )", text):
        if current and len(current) + len(paragraph) > max_chars:
            pieces.append(current)
            current = ""
        current = f"{current}\n{paragraph}" if current else paragraph
    pieces.append(current)
    # A single paragraph can still be too long (e.g. the list of definitions): cut it at line breaks.
    result = []
    for piece in pieces:
        while len(piece) > max_chars:
            cut = piece.rfind("\n", 0, max_chars)
            cut = cut if cut > 0 else max_chars
            result.append(piece[:cut])
            piece = piece[cut:]
        result.append(piece)
    return [p.strip() for p in result if p.strip()]


def make_chunks(articles: list[dict], max_chars: int = MAX_CHARS) -> list[dict]:
    chunks = []
    for a in articles:
        header = f"CRR Article {a['article']} — {a['article_title']} ({a.get('part', '')})"
        for i, piece in enumerate(split_article(a["text"], max_chars)):
            chunks.append({"id": f"{a['article']}#{i}", "article": a["article"], "title": a["article_title"], "part": a.get("part", ""), "text": f"{header}\n{piece}", "url": a["url"]})
    return chunks


chunks = make_chunks(articles)
print(len(chunks), "chunks; length: mean", int(np.mean([len(c["text"]) for c in chunks])), "max", max(len(c["text"]) for c in chunks))
print(chunks[150]["text"][:400])

## 4. Embeddings

An **embedding** model turns a text into a vector (here 1,536 or 3,072 numbers). Texts with similar meaning get vectors that point in similar directions, measured with the **cosine similarity** (1 = same direction, 0 = unrelated).

In [ ]:
def embed(texts: list[str], batch_size: int = 100) -> np.ndarray:
    """Embed texts in batches (Gemini accepts at most 100 per call) and normalise them to length 1."""
    vectors = []
    for i in range(0, len(texts), batch_size):
        response = client.embeddings.create(model=EMBEDDING_MODEL, input=texts[i : i + batch_size])
        vectors += [d.embedding for d in response.data]
    v = np.array(vectors)
    return v / np.linalg.norm(v, axis=1, keepdims=True)  # with unit vectors, cosine similarity = dot product


phrases = [
    "capital conservation buffer",
    "colchón de conservación de capital",
    "a bank must hold extra CET1 capital in good times",
    "mortgage interest rate for first-time buyers",
]
V = embed(phrases)
pd.DataFrame(V @ V.T, index=phrases, columns=[p[:25] for p in phrases]).round(2)

The Spanish phrase is close to its English translation, and the paraphrase is closer to the buffer than the mortgage sentence is, although they share almost no words. That is what lets us search by **meaning**.

Now embed every chunk. The matrix of vectors **is** our vector store; a vector database (Chroma, pgvector, Elasticsearch…) does the same operations at scale, with filters and persistence.

In [ ]:
t = time.time()
E = embed([c["text"] for c in chunks])
print(E.shape, f"in {time.time() - t:.0f}s")

## 5. Search

### 5.1 Semantic search

In [ ]:
def semantic_search(query: str, k: int = 50) -> list[int]:
    """Indices of the k chunks most similar to the query."""
    q = embed([query])[0]
    return list(np.argsort(-(E @ q))[:k])


def top_articles(indices: list[int], k: int = 5) -> list[str]:
    """First k distinct articles among the retrieved chunks."""
    seen = []
    for i in indices:
        if chunks[i]["article"] not in seen:
            seen.append(chunks[i]["article"])
        if len(seen) == k:
            break
    return seen


def show(indices: list[int], k: int = 5) -> None:
    for i in indices[:k]:
        print(f"Art. {chunks[i]['article']:>5} · {chunks[i]['title'][:70]}")


show(semantic_search("Is there a cap on how much a bank can lend to one borrower?"))

In [ ]:
show(semantic_search("What does Article 412 require?"))

The first query finds Article 395 (*Limits to large exposures*) without sharing a single keyword with it: *cap* vs *limits*, *lend* vs *exposure*, *borrower* vs *client*. The second fails: "Article 412" is an **identifier**, and meaning does not help with identifiers, codes or exact terms (`LCR`, `EBA/GL/2020/06`, `Article 412`).

### 5.2 Keyword search (BM25)

The classic search-engine ranking: rewards chunks containing the query's words, especially rare ones.

In [ ]:
from rank_bm25 import BM25Okapi


def tokenize(text: str) -> list[str]:
    return re.findall(r"\w+", text.lower())


bm25 = BM25Okapi([tokenize(c["text"]) for c in chunks])


def keyword_search(query: str, k: int = 50) -> list[int]:
    return list(np.argsort(-bm25.get_scores(tokenize(query)))[:k])


show(keyword_search("What does Article 412 require?"))

### 5.3 Hybrid search

Combine both rankings with **reciprocal rank fusion**: each chunk scores `1 / (60 + rank)` in each list, and the scores are added. It needs no calibration between the two kinds of scores.

In [ ]:
def hybrid_search(query: str, k: int = 50) -> list[int]:
    scores = {}
    for ranking in (semantic_search(query), keyword_search(query)):
        for rank, i in enumerate(ranking):
            scores[i] = scores.get(i, 0) + 1 / (60 + rank + 1)
    return sorted(scores, key=lambda i: -scores[i])[:k]


show(hybrid_search("What does Article 412 require?"))

### 5.4 Which one is better? Measure it

A handful of questions whose correct article we know is enough to compare methods: does the right article appear among the top 5?

In [ ]:
GOLDEN = [
    ("What minimum CET1, Tier 1 and total capital ratios must institutions meet?", {"92"}),
    ("How is the leverage ratio calculated?", {"429"}),
    ("What is the liquidity coverage requirement?", {"412"}),
    ("Which items qualify as Common Equity Tier 1?", {"26"}),
    ("What is the limit on an institution's exposure to a single client or group of connected clients?", {"395"}),
    ("What must be deducted from Common Equity Tier 1 items?", {"36"}),
    ("What risk weight applies to exposures secured by residential property?", {"125", "124"}),
    ("What does Article 412 require?", {"412"}),
    ("What is the net stable funding ratio?", {"428b", "413"}),
    ("What is the output floor and what percentage applies?", {"92", "465"}),
    ("How does the regulation define an institution?", {"4"}),
    ("Do SME exposures get a reduced own funds requirement?", {"501"}),
]


def evaluate(search) -> pd.DataFrame:
    rows = []
    for q, expected in GOLDEN:
        found = top_articles(search(q))
        rows.append({"question": q, "expected": ", ".join(sorted(expected)), "top 5": ", ".join(found), "hit": bool(expected & set(found))})
    return pd.DataFrame(rows)


results = {name: evaluate(f) for name, f in [("semantic", semantic_search), ("keyword", keyword_search), ("hybrid", hybrid_search)]}
pd.DataFrame({name: r["hit"].map({True: "✓", False: "✗"}) for name, r in results.items()}).set_index(results["semantic"]["question"].str[:60])

In [ ]:
{name: f"{r['hit'].sum()}/{len(r)}" for name, r in results.items()}

No method wins everywhere: keyword search catches identifiers, semantic search catches paraphrases, and hybrid trades a bit of both. In a real project you build a **larger** question set with the business (50–100 questions), and every change — chunk size, embedding model, hybrid weights, reranking — is decided by that number, not by intuition.

For identifiers there is an even simpler answer: if the user names an article, **look it up directly**. That will be a separate tool for the agent.

## 6. Generation with citations

Retrieve, put the chunks in the prompt, and ask for an answer that uses **only** them and cites them. The sources are returned by our code, not written by the model, so they cannot be invented.

In [ ]:
RAG_PROMPT = """Answer the question using only the CRR excerpts below.
Cite the article for every statement, like (Art. 92(1)). If the excerpts do not contain the answer, say so.

{context}

Question: {question}"""


def answer_with_sources(question: str, k: int = 5) -> tuple[str, list[dict]]:
    hits = hybrid_search(question)[:k]
    context = "\n\n---\n\n".join(chunks[i]["text"] for i in hits)
    response = client.chat.completions.create(model=MODEL, messages=[{"role": "user", "content": RAG_PROMPT.format(context=context, question=question)}])
    sources = list({chunks[i]["article"]: {"article": chunks[i]["article"], "title": chunks[i]["title"], "url": chunks[i]["url"]} for i in hits}.values())
    return response.choices[0].message.content, sources


answer, sources = answer_with_sources(question)
print(answer, "\n\nSources:")
for s in sources:
    print(f"Art. {s['article']} — {s['title']}: {s['url']}")

Two things to look at:

- **Retrieval.** The fully phased-in percentage is in Article 92(3). If Article 92 is not among the sources, the search missed it, and the honest answer is "the excerpts do not say". Bad retrieval means a bad answer, whatever the prompt.
- **Citations.** If the answer cites an article that is **not** among the sources, the model answered from memory, which the prompt forbade. It happens, so check it in code rather than trusting the prompt:

In [ ]:
def check_citations(answer: str, sources: list[dict]) -> dict:
    cited = set(re.findall(r"Art(?:icle|\.)\s*(\d+[a-z]*)", answer))
    retrieved = {s["article"] for s in sources}
    return {"cited": sorted(cited), "retrieved": sorted(retrieved), "cited but not retrieved": sorted(cited - retrieved)}


check_citations(answer, sources)

In [ ]:
answer, sources = answer_with_sources("What dress code do bank employees have to follow?")
print(answer)

## 7. RAG as a tool for the agent

In notebook 02 the agent read the bank's data. Now give it the regulation too, with two tools:

- `search_regulation(query)`: hybrid search, for questions in natural language;
- `get_article(number)`: direct lookup, for when an article is named.

The agent decides when to search, can search several times, and can combine regulation with the bank's figures.

In [ ]:
from langchain.agents import create_agent
from langchain.tools import tool

model = utils.get_chat_model(PROVIDER, MODEL)
db = utils.load_bank_db()
ARTICLES = {a["article"]: a for a in articles}


@tool(parse_docstring=True)
def search_regulation(query: str) -> list[dict]:
    """Search the CRR (EU capital requirements regulation) and return the 5 most relevant excerpts with their article numbers.

    Args:
        query: What to look for, in natural language.
    """
    return [{"article": chunks[i]["article"], "title": chunks[i]["title"], "text": chunks[i]["text"]} for i in hybrid_search(query)[:5]]


@tool(parse_docstring=True)
def get_article(number: str) -> dict:
    """Get the full text of one CRR article. Use it when the article number is known.

    Args:
        number: Article number, e.g. "92" or "428b".
    """
    a = ARTICLES.get(number.strip())
    if a is None:
        return {"error": f"Article {number} not found. Numbers look like 92 or 428b."}
    return {"article": a["article"], "title": a["article_title"], "text": a["text"][:12000], "url": a["url"]}


@tool(parse_docstring=True)
def get_capital_ratios(reference_date: str) -> dict:
    """Get Demo Bank's capital and liquidity ratios (CET1, Tier 1, total capital, leverage, LCR) at a quarter end.

    Args:
        reference_date: Quarter-end date in YYYY-MM-DD format, e.g. 2026-09-30.
    """
    row = db.execute("SELECT * FROM capital_position WHERE reference_date = ?", (reference_date,)).fetchone()
    if row is None:
        return {"error": f"No data for {reference_date}."}
    rwa = row["rwa_credit_eur_m"] + row["rwa_market_eur_m"] + row["rwa_operational_eur_m"]
    tier1 = row["cet1_capital_eur_m"] + row["at1_capital_eur_m"]
    return {
        "reference_date": reference_date,
        "cet1_ratio_pct": round(100 * row["cet1_capital_eur_m"] / rwa, 2),
        "tier1_ratio_pct": round(100 * tier1 / rwa, 2),
        "total_capital_ratio_pct": round(100 * (tier1 + row["tier2_capital_eur_m"]) / rwa, 2),
        "leverage_ratio_pct": round(100 * tier1 / row["leverage_exposure_eur_m"], 2),
        "lcr_pct": round(100 * row["hqla_eur_m"] / row["net_outflows_30d_eur_m"], 1),
    }


regulation_agent = create_agent(
    model,
    tools=[search_regulation, get_article, get_capital_ratios],
    system_prompt=(
        "You are a regulatory assistant for Demo Bank. Answer from the tools only: the CRR for rules, "
        "the bank's data for figures. Cite articles as (Art. 92(1)). If the tools do not answer, say so."
    ),
)

for q in [
    "Does Demo Bank meet the CRR minimum own funds ratios at 30 September 2026?",
    "What does Article 429 say about how the leverage ratio is calculated? Is our leverage ratio above the minimum?",
]:
    result = regulation_agent.invoke({"messages": [{"role": "user", "content": q}]})
    calls = [c["name"] + str(c["args"]) for m in result["messages"] for c in getattr(m, "tool_calls", [])]
    print(f"> {q}\nTools: {calls}\n\n{result['messages'][-1].text}\n")

## 8. Exercise: improve the retrieval

Each step changes one thing; re-run the evaluation of section 5.4 after each to see the effect.

1. **Your own questions.** Add three questions to `GOLDEN` about topics you know (large exposures, liquidity, disclosure…), with the article you expect. Look it up in `df` if needed. How do the three methods do on them?
2. **Chunk size.** Rebuild the chunks with `MAX_CHARS = 1000` and with `MAX_CHARS = 6000` (you need to re-embed and rebuild BM25). Which works best on the golden set, and how many chunks does each give?
3. **Filter by part.** Add an optional `part` argument to `search_regulation` (e.g. `"PART SIX"`) that keeps only chunks of that part. Ask the agent a liquidity question: does it use the filter?
4. **Bonus — rerank with the LLM.** Take the top 15 chunks of the hybrid search and ask the model to return the 5 most relevant chunk numbers for the question. Does the golden-set score improve? What does it cost in time and tokens?

In [ ]:
# 1. Your own questions
# GOLDEN += [("...", {"..."}), ...]

In [ ]:
# 2. Chunk size

In [ ]:
# 3. Filter by part

In [ ]:
# 4. Bonus: LLM reranking

## Takeaways

- RAG = **retrieve** relevant passages + **augment** the prompt + **generate** with citations. The model answers from *your* documents, and the sources come from your code.
- **Chunking** should follow the document's structure, and every chunk should say where it comes from.
- **Semantic** search finds meaning, **keyword** search finds identifiers; **hybrid** combines them. Exact references deserve a direct lookup.
- Decide with a **golden set**, not with intuition. If retrieval fails, no prompt will fix the answer.
- As a **tool**, retrieval becomes one more thing the agent can do — next to the bank's data.

---
### Solutions to exercise 8

In [ ]:
# 1. Examples (check the expected articles in df before trusting them)
GOLDEN_EXTRA = [
    ("What counts as a large exposure?", {"392"}),
    ("What must institutions report on their liquid assets?", {"416"}),
    ("Which information on own funds must institutions disclose?", {"437"}),
]
GOLDEN += GOLDEN_EXTRA
{name: f"{evaluate(f)['hit'].sum()}/{len(GOLDEN)}" for name, f in [("semantic", semantic_search), ("keyword", keyword_search), ("hybrid", hybrid_search)]}

In [ ]:
# 2. Chunk size: rebuild everything the searches use, then evaluate.
for max_chars in [1000, 6000, MAX_CHARS]:
    chunks = make_chunks(articles, max_chars)
    E = embed([c["text"] for c in chunks])
    bm25 = BM25Okapi([tokenize(c["text"]) for c in chunks])
    scores = {name: evaluate(f)["hit"].sum() for name, f in [("semantic", semantic_search), ("keyword", keyword_search), ("hybrid", hybrid_search)]}
    print(max_chars, len(chunks), "chunks", scores)
# The last iteration restores the default chunks.

In [ ]:
# 3. Filter by part
@tool(parse_docstring=True)
def search_regulation(query: str, part: str = "") -> list[dict]:
    """Search the CRR (EU capital requirements regulation) and return the 5 most relevant excerpts with their article numbers.

    Args:
        query: What to look for, in natural language.
        part: Optional part of the CRR to search in, e.g. "PART SIX" (liquidity) or "PART SEVEN" (leverage). Empty for all.
    """
    hits = [i for i in hybrid_search(query, k=200) if chunks[i]["part"].upper().startswith(part.upper())][:5]
    return [{"article": chunks[i]["article"], "title": chunks[i]["title"], "text": chunks[i]["text"]} for i in hits]


regulation_agent = create_agent(model, tools=[search_regulation, get_article, get_capital_ratios], system_prompt="You are a regulatory assistant for Demo Bank. Answer from the tools only and cite articles.")
result = regulation_agent.invoke({"messages": [{"role": "user", "content": "Which assets qualify as level 1 liquid assets for the LCR? Search only the liquidity part."}]})
print([c["args"] for m in result["messages"] for c in getattr(m, "tool_calls", [])])
print(result["messages"][-1].text)

In [ ]:
# 4. Bonus: LLM reranking
RERANK_PROMPT = """Question: {question}

Below are numbered excerpts of the CRR. Return the numbers of the 5 most relevant excerpts for answering the question,
most relevant first, as a comma-separated list and nothing else.

{excerpts}"""


def reranked_search(query: str, k: int = 50) -> list[int]:
    candidates = hybrid_search(query)[:15]
    excerpts = "\n\n".join(f"[{n}] {chunks[i]['text'][:800]}" for n, i in enumerate(candidates))
    reply = client.chat.completions.create(model=MODEL, messages=[{"role": "user", "content": RERANK_PROMPT.format(question=query, excerpts=excerpts)}])
    order = [int(n) for n in re.findall(r"\d+", reply.choices[0].message.content) if int(n) < len(candidates)]
    ranked = [candidates[n] for n in dict.fromkeys(order)]
    return ranked + [i for i in candidates if i not in ranked]


t = time.time()
print("reranked:", f"{evaluate(reranked_search)['hit'].sum()}/{len(GOLDEN)}", f"in {time.time() - t:.0f}s")